In [1]:
import numpy as np
import pandas as pd
import re
from pathlib import Path

In [2]:
DATA = Path("../data")

lyrics_df = pd.read_csv(DATA / 'lyrics_transformed.csv').drop('Unnamed: 0', axis=1)
audio_df = pd.read_csv(DATA / 'audio_features_transformed.csv').drop('Unnamed: 0', axis=1)

Currently, the formatting of song names in the dataframes are different. We make these consistent with a function.

In [3]:
def transform_name(name):
    #get rid of spaces
    name = re.sub(r" ", "", name)

    #get rid of apostrophes and +
    name = re.sub(r"['’‘+\"]", "", name)

    #change punctuation to underscore
    name = re.sub(r'[()?.,\-!&]', "_", name)

    #lowercase all
    name = name.lower()

    #remove 'bonus track'
    name = re.sub(r"_bonustrack", "", name)
    
    #remove features
    name = re.sub(r'_feat_[a-z]+_', "", name)

    #final exceptions
    name = re.sub(r"popversion_", "popversion", name)
    name = re.sub(r"atthedisco_", "", name)
    
    return name

In [4]:
# Create normalised names
audio_df["norm_name"] = audio_df["name"].apply(transform_name)
lyrics_df["norm_name"] = lyrics_df["name"].apply(transform_name)

audio_df = audio_df.rename(columns={'name': 'audio_name'})
lyrics_df = lyrics_df.rename(columns={'name': 'lyrics_name'})

# drop duplicates (other artist features)
audio_df = audio_df.drop([52, 96, 97])
lyrics_df = lyrics_df.drop(207)

# join DataFrames on the normalised name
merged = audio_df.merge(lyrics_df, on="norm_name", how="inner")
merged["id"] = pd.factorize(merged["norm_name"])[0]

# for part 4: A/B testing
merged.to_csv('merged.csv', index=False)

In [ ]:
# create lookup tables for efficiency
name_to_id = (
    merged[["id", "audio_name"]]
    .rename(columns={"audio_name": "name"})
    .drop_duplicates()
    .sort_values("id")
    .set_index('name')
)

id_to_name = name_to_id.reset_index().set_index("id")

# for streamlit deployment
#name_to_id.to_csv('name_to_id.csv')

Dissimilarity matrices can be calculated. Euclidean distance is used for audio features for consistency with original project. Cosine similarity is used for lyrics for better accuracy with semantics.

In [ ]:
lyrics_df = merged[lyrics_df.columns].drop(['lyrics_name', 'norm_name'], axis=1)
audio_df = merged[audio_df.columns].drop(['audio_name', 'album', 'norm_name'], axis=1)

from sklearn.metrics import pairwise_distances

D_lyrics = pairwise_distances(lyrics_df, metric='cosine')
D_audio = pairwise_distances(audio_df, metric='euclidean')

# This needs scaling - dividing both by max will scale distances to [0, 1], since they are non-negative
audio_max, lyrics_max = np.max(D_audio), np.max(D_lyrics)

# convert matrices to DataFrames and scale
D_audio_df = pd.DataFrame(D_audio, columns=[i for i in range(len(merged))])
D_lyrics_df = pd.DataFrame(D_lyrics, columns=[i for i in range(len(merged))])

D_audio_df = D_audio_df/audio_max
D_lyrics_df = D_lyrics_df/lyrics_max

# for streamlit deployment
#D_audio_df.to_csv('audio_dissimilarities.csv', index=False)
#D_lyrics_df.to_csv('lyrics_dissimilarities.csv', index=False)

D_audio_df.head()

,0,1,2,3,4,5,6,7,8,9,...,222,223,224,225,226,227,228,229,230,231
0,0.000000,0.396425,0.491853,0.288758,0.403041,0.572610,0.436796,0.480232,0.402396,0.387742,...,0.632782,0.465504,0.553487,0.453771,0.633366,0.611473,0.535236,0.585507,0.503076,0.519160
1,0.396425,0.000000,0.328196,0.414916,0.556929,0.412791,0.430403,0.501950,0.275410,0.227174,...,0.504571,0.485983,0.444993,0.429252,0.472370,0.432847,0.517576,0.498324,0.490466,0.660430
2,0.491853,0.328196,0.000000,0.543384,0.658384,0.506448,0.420690,0.422330,0.365586,0.460761,...,0.410142,0.608700,0.471947,0.439762,0.427137,0.260575,0.537130,0.483814,0.539993,0.666275
3,0.288758,0.414916,0.543384,0.000000,0.191654,0.391695,0.421308,0.382292,0.426502,0.260782,...,0.682694,0.502515,0.620388,0.547347,0.652732,0.637867,0.515023,0.613984,0.544496,0.584895
4,0.403041,0.556929,0.658384,0.191654,0.000000,0.413394,0.437545,0.431608,0.531795,0.391939,...,0.753877,0.551018,0.697879,0.618902,0.703486,0.733797,0.544497,0.694312,0.602710,0.595409


Recommendation algorithm 1: k-NN approach. Calculate a convex combination of dissimilarities, and return the k smallest dissimilarities, ie, $$D_{combined} := \alpha D_{audio} + (1-\alpha) D_{lyrics}$$ for some $\alpha$.

Different recommendations could be offered by varying $\alpha$, for example, 0.3 for lyric-heavy, 0.5 for balanced, 0.7 for music-heavy.

In [7]:
def recommend(query_name, alpha=0.5, k=5):
    D = alpha * D_audio_df + (1 - alpha) * D_lyrics_df
    query_id = name_to_id.loc[query_name, 'id']
    distances = D.loc[query_id].copy()
    distances.loc[query_id] = np.inf  # don't recommend itself
    return [id_to_name.loc[x]['name'] for x in distances.nsmallest(k).index]

recs = []
alphas = []
for alpha in np.arange(0, 1.1, 0.1):
    alphas.append(alpha)
    recs.append(recommend("Style (Taylor's Version)", alpha=alpha))

alpha_df = pd.DataFrame({
    'alpha': alphas, 
    'rec_1': [recs[i][0] for i in range(len(recs))],
    'rec_2': [recs[i][1] for i in range(len(recs))],
    'rec_3': [recs[i][2] for i in range(len(recs))],
    'rec_4': [recs[i][3] for i in range(len(recs))],
    'rec_5': [recs[i][4] for i in range(len(recs))]
})
alpha_df

,alpha,rec_1,rec_2,rec_3,rec_4,rec_5
0,0.0,Tim McGraw,I Wish You Would (Taylor's Version),Wildest Dreams (Taylor's Version),Cornelia Street,So High School
1,0.1,I Wish You Would (Taylor's Version),Tim McGraw,Wildest Dreams (Taylor's Version),Cornelia Street,So High School
2,0.2,I Wish You Would (Taylor's Version),Tim McGraw,Wildest Dreams (Taylor's Version),Cornelia Street,So High School
3,0.3,I Wish You Would (Taylor's Version),Tim McGraw,Wildest Dreams (Taylor's Version),Begin Again (Taylor's Version),‘tis the damn season
4,0.4,I Wish You Would (Taylor's Version),Wildest Dreams (Taylor's Version),Tim McGraw,Begin Again (Taylor's Version),"""Slut!"" (Taylor's Version) (From The Vault)"
5,0.5,I Wish You Would (Taylor's Version),Wildest Dreams (Taylor's Version),Begin Again (Taylor's Version),Tim McGraw,Say Don't Go (Taylor's Version) (From The Vault)
6,0.6,I Wish You Would (Taylor's Version),Wildest Dreams (Taylor's Version),Begin Again (Taylor's Version),Tim McGraw,Is It Over Now? (Taylor's Version) (From The V...
7,0.7,Begin Again (Taylor's Version),I Wish You Would (Taylor's Version),Wildest Dreams (Taylor's Version),Is It Over Now? (Taylor's Version) (From The V...,When Emma Falls in Love (Taylor’s Version) (Fr...
8,0.8,When Emma Falls in Love (Taylor’s Version) (Fr...,Begin Again (Taylor's Version),Is It Over Now? (Taylor's Version) (From The V...,Come Back...Be Here (Taylor's Version),I Wish You Would (Taylor's Version)
9,0.9,When Emma Falls in Love (Taylor’s Version) (Fr...,Come Back...Be Here (Taylor's Version),Is It Over Now? (Taylor's Version) (From The V...,Begin Again (Taylor's Version),Sparks Fly (Taylor’s Version)


Recommendation algorithm 2: Historical searches can also be considered in a similar k-NN approach. The current search should dominate the results, while previous searches can be weighted by considering recency.

Let $D_{current}$ be the dissimilarity vector of the current search, calculated as above, and $D_0, D_1, ..., D_{N-1}$ be the $N$ vectors of ordered historical searches, most recent first. Let $W$ be scalar weights named similarly.

Then $$D := W_{current} \cdot D_{current} + \sum_{i=0}^{N-1} \left(W(i, N) \cdot D_i \right)$$ where $W_{current} + \sum_{i} W(i, N) = 1$.

Let $W_{current} = 0.8$. Then the weight function $W(i, N) = \frac{2(N-i)}{5N(N+1)}$ satisfies $\sum_{i} W(i, N) = 0.2$, while also being a decreasing function of $i$ for constant $N$. 

In general, the weight function $W(i, N) = \frac{2(1-W_{current})(N-i)}{N(N+1)}$ is valid.

In [8]:
def recommend_weighted(query_name: str, prev_queries: list, alpha=0.5, k=5):
    """Takes query and a list of previous queries (ordered, most recent first)"""

    D = alpha * D_audio_df + (1 - alpha) * D_lyrics_df

    # weight function for current query
    if prev_queries:
        s_current = 0.8
    else:
        s_current = 1

    query_id = name_to_id.loc[query_name, 'id']
    distances = D.loc[query_id].copy()*s_current
    distances.loc[query_id] = np.inf  # don't recommend itself

    # define weight function for previous queries
    def W(i, S):
        return (2*(S-i))/(5*S*(S+1))
        
    for i, query in enumerate(prev_queries):
        query_id = name_to_id.loc[query, 'id']
        query_distances = D.loc[query_id].copy() * W(i, len(prev_queries))
        query_distances.loc[query_id] += 1/(i+1) # make it less likely to recommend something already searched 
        distances = distances + query_distances
    return [id_to_name.loc[x]['name'] for x in distances.nsmallest(k).index]

recommend_weighted("Style (Taylor's Version)", ["The Archer", "champagne problems", "I Wish You Would (Taylor's Version)"], alpha=0.5)


["Wildest Dreams (Taylor's Version)",
 "Begin Again (Taylor's Version)",
 'Tim McGraw',
 "Say Don't Go (Taylor's Version) (From The Vault)",
 "State Of Grace (Taylor's Version)"]

Both recommendation algorithms can be combined into a single user-friendly search function with a boolean flag.

In [9]:
def search(query_name: str, prev_queries: list, balance='equal', use_prev=True, k=5):
    """
    Returns top 5 recommendations given query name and list of previous queries. 
    balance takes values in {'equal', 'lyrics', 'audio'}, changing the fusion weights of the dissimilarity matrices.
    use_prev is boolean. True means previous searches are used to enhance recommendations.
    """

    balance_to_alpha = {
        'equal': 0.5,
        'lyrics': 0.3, 
        'audio': 0.7
    }

    assert type(use_prev) == bool
    if use_prev == True:
        return recommend_weighted(query_name, prev_queries, alpha=balance_to_alpha[balance], k=k)
    else:
        return recommend(query_name, alpha=balance_to_alpha[balance], k=k)
    
search('the last great american dynasty', ['Gorgeous', 'Lover'], balance='audio', use_prev=True, k=5)

["Speak Now (Taylor's Version)",
 'mad woman',
 "Starlight (Taylor's Version)",
 'long story short',
 'Timeless (Taylor’s Version) (From The Vault)']

In [10]:
search('the last great american dynasty', [], balance='audio', use_prev=True, k=5)

["Speak Now (Taylor's Version)",
 'mad woman',
 'Timeless (Taylor’s Version) (From The Vault)',
 'long story short',
 "Starlight (Taylor's Version)"]

The system using previous search history ranks 'Starlight' higher, due to 'Gorgeous' also being an upbeat track.

Since this recommendation system is unsupervised, metrics like precision @ K and recall @ K cannot be used. However, we can measure the coverage of data in the top 5 of all searches, and examine the relevance of the recommendations given.

In [11]:
top_five = []
for song in name_to_id.index:
    results = search(song, [], balance='lyrics', use_prev=False, k=5)
    for result in results:
        top_five.append(result)

print(f'{round(len(set(top_five))*100/len(name_to_id.index), 2)}% coverage')

98.71% coverage


In [12]:
import random
random.seed(42)
for song in random.sample(list(name_to_id.index), 5):
    print((song, search(song, [], balance='equal', use_prev=True, k=5)))

('marjorie', ['coney island (feat. The National)', 'invisible string', 'evermore (feat. Bon Iver)', 'dorothea', 'tolerate it'])
('The Bolter', ['The Manuscript', 'The Prophecy', 'The Albatross', 'The Black Dog', 'So Long, London'])
('Fresh Out The Slammer', ['But Daddy I Love Him', "Style (Taylor's Version)", 'The Manuscript', 'Tim McGraw', 'When Emma Falls in Love (Taylor’s Version) (From The Vault)'])
('The Archer', ['tolerate it', 'epiphany', 'exile (feat. Bon Iver)', 'long story short', 'my tears ricochet'])
('I Can See You (Taylor’s Version) (From The Vault)', ["Girl At Home (Taylor's Version)", 'Love Story (Taylor’s Version)', 'Paper Rings', "The Story Of Us (Taylor's Version)", "How You Get The Girl (Taylor's Version)"])


1) recommended 4 other evermore songs and 1 folklore song, which were released in the same year and have the same musical style
2) recommended 5 other songs from the same album
3) recommended 2 songs from the same album, 3 songs with love themes
4) recommended 4 other sad songs ('tolerate it' and 'my tears ricochet' are also track 5, even though track number was not used as a feature), hints of hope similar to long story short
5) recommended 5 other upbeat songs

The recommendation system seems to be relevant (random samples) and diverse (98.71% coverage).